In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ---------------- 路径配置 ----------------
RESULT_DIR = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"
CLIM_206YR_NC = os.path.join(RESULT_DIR, "EPFLUX_206yr_Climatology_Stats.nc")
DATA_23YR_NC  = os.path.join(RESULT_DIR, "EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc")
OUT_FIG = os.path.join(RESULT_DIR, "Fz_Climatology_Comparison_206yr_vs_23yr.png")

# ---------------- 参数设置 ----------------
LAT0, LAT1 = 40.0, 80.0

def get_40_80_mean(da):
    """计算 40-80N 的面积加权平均"""
    lat = da["lat"]
    # 自动处理纬度顺序
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(LAT1, LAT0) if dec else slice(LAT0, LAT1)
    sub = da.sel(lat=slc)
    weights = np.cos(np.deg2rad(sub["lat"]))
    return sub.weighted(weights).mean("lat")

# ---------------- 1. 处理 206年气候态数据 ----------------
print("Loading 206-year climatology...")
ds_206 = xr.open_dataset(CLIM_206YR_NC)
# 注意：Code A 存的是向下为正，这里取反变为向上为正
fz_mean_206 = -ds_206["Fz_mean"]
fz_std_206  = ds_206["Fz_std"]
plev_206    = ds_206["plev"]

# ---------------- 2. 处理 23年气候态数据 ----------------
print("Processing 23-year data to get climatology...")
ds_23 = xr.open_dataset(DATA_23YR_NC)
fz_raw_23 = ds_23["EP2_cart"]

# 区域平均 (time, plev)
fz_23_zonal = get_40_80_mean(fz_raw_23)

# 筛选冬半年 (10月-3月)
mon = fz_23_zonal["time"].dt.month
mask = (mon >= 10) | (mon <= 3)
fz_23_winter = fz_23_zonal.where(mask, drop=True)

# 计算均值和标准差 (向上为正)
fz_mean_23 = -fz_23_winter.mean("time")
fz_std_23  = fz_23_winter.std("time")
plev_23    = ds_23["plev"]

# ---------------- 3. 开始绘图 ----------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6), sharey=True)

# --- 左图：206年气候态 ---
ax1.plot(fz_mean_206, plev_206/100, 'r-', linewidth=2, label='Mean')
ax1.fill_betweenx(plev_206/100, 
                  (fz_mean_206 - fz_std_206), 
                  (fz_mean_206 + fz_std_206), 
                  color='red', alpha=0.2, label=r'1$\sigma$ Range')
ax1.set_title("206-yr Climatology (Oct-Mar)\n40–80°N Mean $F_z$", fontsize=12)
ax1.set_xlabel("$F_z$ [hPa m s$^{-2}$]")
ax1.set_ylabel("Pressure [hPa]")
ax1.set_yscale('log')
ax1.invert_yaxis()
ax1.grid(True, which="both", ls="--", alpha=0.5)
ax1.legend()

# --- 右图：23年气候态 ---
ax2.plot(fz_mean_23, plev_23/100, 'b-', linewidth=2, label='Mean')
ax2.fill_betweenx(plev_23/100, 
                  (fz_mean_23 - fz_std_23), 
                  (fz_mean_23 + fz_std_23), 
                  color='blue', alpha=0.2, label=r'1$\sigma$ Range')
ax2.set_title("23-yr Climatology (Oct-Mar)\n40–80°N Mean $F_z$", fontsize=12)
ax2.set_xlabel("$F_z$ [hPa m s$^{-2}$]")
ax2.grid(True, which="both", ls="--", alpha=0.5)
ax2.legend()

# 统一设置 Y 轴刻度
yticks = [1000, 700, 500, 300, 200, 100, 50, 30, 10]
ax1.set_yticks(yticks)
ax1.set_yticklabels([str(t) for t in yticks])

plt.tight_layout()
plt.savefig(OUT_FIG, dpi=200)
print(f"Comparison plot saved to: {OUT_FIG}")
plt.show()

# 资源释放
ds_206.close()
ds_23.close()

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ---------------- 路径配置 ----------------
RESULT_DIR = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result"
DATA_23YR_NC = os.path.join(RESULT_DIR, "EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc")
OUT_FIG = os.path.join(RESULT_DIR, "Fz_Winter_Evolution_Climatology_23yr_1_100hPa.png")

# ---------------- 参数设置 ----------------
LAT0, LAT1 = 40.0, 80.0
P_MIN, P_MAX = 100.0, 10000.0  # 单位 Pa (对应 1hPa 到 100hPa)

# ---------------- 1. 加载并处理数据 ----------------
print("Loading 23-year daily dataset...")
ds = xr.open_dataset(DATA_23YR_NC)

# 提取 Fz 并取反（向上为正）
fz_raw = -ds["EP2_cart"]

# 筛选气压层 1-100hPa
fz_raw = fz_raw.sel(plev=slice(P_MIN, P_MAX))

# 计算 40-80°N 面积加权平均
lat = fz_raw["lat"]
dec = float(lat.values[0]) > float(lat.values[-1])
fz_sub = fz_raw.sel(lat=slice(LAT1, LAT0) if dec else slice(LAT0, LAT1))
weights = np.cos(np.deg2rad(fz_sub["lat"]))
fz_area_mean = fz_sub.weighted(weights).mean("lat")  # (time, plev)

# ---------------- 2. 计算日气候态 (Daily Climatology) ----------------
print("Calculating daily climatology (23-year mean)...")
fz_daily_clim = fz_area_mean.groupby("time.dayofyear").mean("time")

# ---------------- 3. 构造冬季时间窗口 (11月1日 - 4月30日) ----------------
# 模式为 365 天制 (NoLeap)
winter_indices = np.concatenate([np.arange(305, 366), np.arange(1, 121)])
fz_plot = fz_daily_clim.sel(dayofyear=winter_indices)

# ---------------- 4. 绘图 ----------------
fig, ax = plt.subplots(figsize=(12, 6))

plev_hpa = fz_plot["plev"] / 100.0
x = np.arange(len(fz_plot.dayofyear))

# 设置填色范围
vmax = float(fz_plot.max()) * 0.9
levels = np.linspace(0, vmax, 21)

cf = ax.contourf(x, plev_hpa, fz_plot.transpose("plev", "dayofyear"), 
                 levels=levels, cmap="YlOrRd", extend="both")

# 轴设置
ax.set_yscale("log")
ax.invert_yaxis()
ax.set_ylim(100, 1)  # 严格限制在 100hPa 到 1hPa
ax.set_ylabel("Pressure (hPa)", fontsize=12)
ax.set_title("23-year Climatological Winter $F_z$ Evolution (1–100 hPa)\n40–80°N Area Mean, Upward Positive", fontsize=14)

# 标注月份起始位置
tick_pos = [0, 30, 61, 92, 120, 151]
tick_lab = ["Nov 1", "Dec 1", "Jan 1", "Feb 1", "Mar 1", "Apr 1"]
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_lab)

# Y 轴刻度 (在 1-100hPa 范围内的常用刻度)
yticks = [100, 70, 50, 30, 20, 10, 7, 5, 3, 2, 1]
ax.set_yticks(yticks)
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())

# 颜色条
cbar = fig.colorbar(cf, ax=ax, pad=0.02)
cbar.set_label("$F_z$ [hPa m s$^{-2}$]")

plt.tight_layout()
plt.savefig(OUT_FIG, dpi=200, bbox_inches="tight")
print(f"Plot saved to: {OUT_FIG}")
plt.show()

ds.close()